In [1]:
# Typical steps
# 1. Application parses policy documents and extracts relevant information to create a knowledge base.

# This data will be stored in vector database and will be used as context for question answering.
documents = [
        "Employees are compensated on a bi-weekly basis through direct deposit.",       
        "Employees must submit a leave request for approval.", 
        "Company internet must be used for work-related tasks only.",
        "Company internet is a broadband internet.",
        "Employees can take an hour break.",
        "Interact with each employee with Respect"
]
# array([0.12655608, 0.09747534, 0.447553  , 0.47982502, 0.11316214,
#        0.04429522],
# content_corpus
content_corpus = documents
content_corpus

['Employees are compensated on a bi-weekly basis through direct deposit.',
 'Employees must submit a leave request for approval.',
 'Company internet must be used for work-related tasks only.',
 'Company internet is a broadband internet.',
 'Employees can take an hour break.',
 'Interact with each employee with Respect']

In [2]:
# Goal: Retrieve relevant documents based on user query (a.k.a. Context Retrieval)

%pip install -q sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6014.22it/s]


In [5]:
doc_vectors

array([[-0.01192885,  0.01236371,  0.03582985, ...,  0.01726842,
         0.07261268, -0.03175012],
       [ 0.03315504,  0.04853384,  0.04736273, ...,  0.10182011,
         0.09159278,  0.00358371],
       [-0.07135905, -0.03066469,  0.03183773, ..., -0.04109798,
         0.06524781, -0.00688533],
       [-0.00383744, -0.02336746,  0.02958675, ..., -0.04415295,
         0.1255909 , -0.03139852],
       [-0.01790441,  0.01495854,  0.08163832, ..., -0.03217229,
        -0.00513653,  0.05279532],
       [-0.00240892,  0.03361144, -0.06162648, ...,  0.04830876,
         0.0370764 , -0.01683051]], shape=(6, 384), dtype=float32)

In [24]:
# Step 3: User Query and Semantic Matching

query = "What’s internet guidelines?"
query_vec = model.encode([query])[0]
query_vec

array([ 9.66182677e-04, -7.41005614e-02, -3.41320895e-02, -7.21673146e-02,
       -3.01459953e-02,  3.15931137e-03,  5.78168035e-03, -1.49820084e-02,
       -7.70179927e-02, -6.71131164e-02,  3.68748009e-02,  5.36350720e-02,
       -1.55234458e-02, -2.69710701e-02,  3.14551848e-03, -3.05456240e-02,
        7.34010339e-02, -6.38115332e-02, -5.40371686e-02, -7.31853768e-02,
        9.77015495e-02, -1.28768384e-02,  1.86218880e-02, -3.70852873e-02,
       -1.30927898e-02, -3.48918103e-02, -1.81941036e-02,  6.01696596e-02,
       -2.58792061e-02,  2.63745780e-03,  1.08246775e-02, -8.39324519e-02,
        7.91806281e-02,  9.32897441e-03, -8.26553181e-02, -3.92252915e-02,
        3.06712948e-02, -4.21725679e-03, -5.92750534e-02,  3.05801630e-02,
        3.55132669e-02, -6.53826818e-02, -2.10071728e-02,  2.03608833e-02,
        7.33901113e-02,  2.84183770e-02, -4.81815748e-02,  2.53268257e-02,
       -5.87922782e-02,  8.54864996e-03,  1.24356737e-02,  2.14819442e-02,
       -2.20838562e-02,  

In [28]:
similarities = model.similarity(query_vec, doc_vectors)
print(type(similarities))
print(f"similarity 1 : {similarities}")

# Ensure it's a 1D numpy array
import numpy as np
similarities = np.asarray(similarities).squeeze()
similarities

<class 'torch.Tensor'>
similarity 1 : tensor([[-0.0432,  0.0881,  0.4001,  0.4128,  0.0236,  0.0750]])


array([-0.04319496,  0.0881235 ,  0.40013427,  0.4127909 ,  0.02355964,
        0.07498439], dtype=float32)

In [26]:
# Now get top 3
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

[3 2 1]


array([0.4127909 , 0.40013427, 0.0881235 ], dtype=float32)

In [27]:
top_docs = [documents[i] for i in top_3_indices]


print (top_docs)
context = ", ".join(top_docs)
context

['Company internet is a broadband internet.', 'Company internet must be used for work-related tasks only.', 'Employees must submit a leave request for approval.']


'Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees must submit a leave request for approval.'